# CNN API demo
See also: `src/cnn/README.md` for instructions on API usage and how to add new models, datasets, etc.

## Use as a template
This notebook can be used as a template for your experiments. You can copy and modify the code to create new experiments with different models, optimizers, etc. The idea is to have a clear and organized structure for your experiments, and to use the config object to easily manage and change the settings for each experiment.

In [ ]:
import copy
import pprint

import torch
import torch.nn as nn

import src.cnn.cnn_paths as paths
from src.cnn.configs import ExperimentConfig
from src.cnn.cnn_models import ImprovedModel
from src.cnn.cnn_registry import build_model
from src.cnn.cnn_utils import (
    load_datasets,
    print_dataset_info,
    make_train_loader,
    make_eval_loader,
    get_device,
    get_num_parameters,
    train_eval,
    evaluate, run_experiment,
)

# Create config object and fill all sections
* **Experiment config** consists of other configs: dataset, model, loader, optimizer, etc.
* The idea is to compile the big experiment config out of smaller ones and to vary the smaller configs for experiments.
> For example, you want to compare **3 different optimizers for the same model** (see the example: [Config change - new experiment](#config_change_new_experiment)).
>
> Then, you can create **3 different optimizer configs**, but keep the rest of **the experiment config the same**. This way, you can easily run experiments with different optimizers while keeping everything else constant. That is the idea of **"controlled experiments"** - you want to change only one variable at a time to see its effect on the outcome.

In [ ]:
# see configs.py
cfg = ExperimentConfig()

# =========================================================
# DATASET CONFIG
# =========================================================
cfg.dataset.dataset_dir = paths.DATASET_DIR # see cnn_paths.py
cfg.dataset.train_subdir = "train"
cfg.dataset.val_subdir = "validate"
cfg.dataset.test_subdir = None   # or "test" if you have it
cfg.dataset.image_size = 224

# leave transforms as None -> default Resize + ToTensor pipeline
cfg.dataset.train_transform = None
cfg.dataset.eval_transform = None

# optional normalization
cfg.dataset.normalize_mean = None
cfg.dataset.normalize_std = None


# =========================================================
# MODEL CONFIG
# IMPORTANT: name is saved in MODEL_REGISTRY -> cnn_registry.py
# use @register_model("depth_cnn") decorator to add new models to the registry and make them available by name in the config
# =========================================================
cfg.model.name = "improved_model"
cfg.model.kwargs = {
    "in_channels": 3,
    "num_classes": 10,
    "units": 128,
    "drop": 0.5,
}


# =========================================================
# DATALOADER CONFIG
# =========================================================
cfg.loader.batch_size = 16
cfg.loader.num_workers = 0
cfg.loader.pin_memory = True
cfg.loader.train_shuffle = True
cfg.loader.eval_shuffle = False
cfg.loader.drop_last_train = False
cfg.loader.drop_last_eval = False


# =========================================================
# LOSS CONFIG
# =========================================================
cfg.loss.cls = nn.CrossEntropyLoss
cfg.loss.kwargs = {}


# =========================================================
# OPTIMIZER CONFIG
# =========================================================
cfg.optimizer.cls = torch.optim.Adam
cfg.optimizer.kwargs = {
    "lr": 1e-3,
    "weight_decay": 1e-4,
}


# =========================================================
# SCHEDULER CONFIG
# Example: Reduce LR when validation loss plateaus
# =========================================================
cfg.scheduler.cls = torch.optim.lr_scheduler.ReduceLROnPlateau
cfg.scheduler.kwargs = {
    "mode": "min",
    "factor": 0.5,
    "patience": 2,
}
cfg.scheduler.step_metric = "val/loss"


# =========================================================
# TRAIN CONFIG
# =========================================================
cfg.train.epochs = 3
cfg.train.device = str(get_device("auto"))
cfg.train.non_blocking = True
cfg.train.use_amp = False
cfg.train.grad_clip_norm = None
cfg.train.best_metric = "val/accuracy"
cfg.train.best_mode = "max"
cfg.train.seed = 13


# =========================================================
# W&B CONFIG
# =========================================================
cfg.wandb.enabled = True
cfg.wandb.project = "MPW-CNN"
cfg.wandb.entity = "MSE_DeLearn_SPR26"
cfg.wandb.mode = "online"   # "online", "offline", or "disabled" for no logging

# NOTE: use meaningful names for runs, so the difference is clear
cfg.wandb.run_name = "demo_improved_model_Adam"

# NOTE: use meaningful grouping, for example, by model or by task.
# E.g. task "Data augmentation" => experiments on light-mid-heavy augmentation
cfg.wandb.group = "demo_runs"
cfg.wandb.job_type = "train"

# NOTE: use meaningful tags to filter runs in UI
cfg.wandb.tags = ["demo", "cnn"]
cfg.wandb.notes = "Full config demo run"

cfg.wandb.log_epoch_metrics = True
cfg.wandb.log_every_n_epochs = 1

cfg.wandb.metric_allowlist = {
    "train/loss",
    "train/accuracy",
    "val/loss",
    "val/accuracy",
    "gap/accuracy",
    "gap/loss",
    "lr",
}

cfg.wandb.summary_allowlist = {
    "best_epoch",
    "best_metric_name",
    "best_metric_value",
    "train_final/loss",
    "train_final/accuracy",
    "val_final/loss",
    "val_final/accuracy",
}

cfg.wandb.watch_model = False
cfg.wandb.watch_log = "all"
cfg.wandb.watch_log_freq = 100

# Data
## Get data from config

In [ ]:
# see cnn_utils.py
datasets_dict = load_datasets(cfg.dataset)

train_dataset = datasets_dict["train"]
val_dataset = datasets_dict["val"]
test_dataset = datasets_dict.get("test")

## Inspect dataset
Sanity check: the results should match.

In [ ]:
print_dataset_info(train_dataset, title="Train dataset")
print()
print_dataset_info(val_dataset, title="Validation dataset")

In [ ]:
print("Train classes:", train_dataset.classes)
print("Val classes:  ", val_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)

## Build dataloaders

In [ ]:
train_loader = make_train_loader(train_dataset, cfg)
val_loader = make_eval_loader(val_dataset, cfg)

test_loader = None
if test_dataset is not None:
    test_loader = make_eval_loader(test_dataset, cfg)

# Model
## Create model from config

In [ ]:
# Model builder from config

model = build_model(cfg.model)

print(model)
print(f"Trainable parameters: {get_num_parameters(model):,}")


# Train and evaluate

In [ ]:
model, history, result = train_eval(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    cfg=cfg,
    run_name=cfg.wandb.run_name,
)

# Training and evaluation results

In [ ]:
pprint.pprint(result)

## History inspection

In [ ]:
print(history.keys())

## Evaluate on loader

In [ ]:
criterion = cfg.loss.cls(**cfg.loss.kwargs).to(torch.device(cfg.train.device))

val_metrics = evaluate(
    model=model,
    loader=val_loader,
    criterion=criterion,
    cfg=cfg,
    prefix="val_manual",
)

print(val_metrics)

# Run experiment: short version of the same logic

In [ ]:
# single method for the whole experiment from config object

model, history, result = run_experiment(cfg)
pprint.pprint(result)

## Config change - new experiment
Here, we simply change optimizer: Adam => SGD.

In [ ]:
# NOTE: deep copy needed for actual new object
cfg2 = copy.deepcopy(cfg)

cfg2.optimizer.cls = torch.optim.SGD
cfg2.wandb.run_name = "demo_improved_model_SGD"

model2, history2, result2 = run_experiment(cfg2)

In [ ]:
pprint.pprint(result2)
